# CLIF 2.1 → 3.0 Column Crosswalk — Demo

This workbook demonstrates `clifpy.crosswalk_table_2_1_to_3_0`, which migrates a site's
**standardized** columns (`*_category`, `*_group`, `*_type`) from CLIF **2.1** to **3.0**.

What it does:
- Lowercases + snake_cases values (`IMV` → `imv`, `Black or African American` → `black_or_african_american`, `l&d` → `l_and_d`).
- Applies curated, non-derivable renames (`High Flow NC` → `hfnc`, `Assist Control-Volume Control` → `acvc`, `Long Term Care Hospital (LTACH)` → `ltach`, `Psychiatric Hospital` → `mental_health_hosp`).
- Leaves un-mappable values **unchanged** and flags them in a report (1→many like `albumin`, or values removed in 3.0 like the `csf` lab specimen).

It is **non-mutating**: it returns a converted *copy* plus a change report. The default table scope is the 16 CLIF beta tables.

In [16]:
import warnings
warnings.filterwarnings("ignore")
import json
import pandas as pd


from clifpy import crosswalk_table_2_1_to_3_0, BETA_TABLES
from clifpy.data import load_demo_clif

# print("Beta tables (default scope):")
# print(", ".join(BETA_TABLES))

In [17]:
# The built-in demo data is in CLIF 2.1 format.
co = load_demo_clif(tables=BETA_TABLES)
print("Loaded demo tables:", co.get_loaded_tables())

ValueError: Unknown table name(s): code_status, crrt_therapy, hospital_diagnosis, medication_admin_intermittent, microbiology_culture, microbiology_susceptibility, patient_procedures. Available demo tables: ['adt', 'hospitalization', 'labs', 'medication_admin_continuous', 'patient', 'patient_assessments', 'position', 'respiratory_support', 'vitals']

In [20]:
from clifpy.schemas import load_schema

schema = load_schema("respiratory_support", "3.0")
schema

{'table_name': 'respiratory_support',
 'version': '3.0',
 'columns': [{'name': 'hospitalization_id',
   'data_type': 'VARCHAR',
   'required': True,
   'is_category_column': False,
   'is_group_column': False},
  {'name': 'recorded_dttm',
   'data_type': 'DATETIME',
   'required': True,
   'is_category_column': False,
   'is_group_column': False},
  {'name': 'device_name',
   'data_type': 'VARCHAR',
   'required': False,
   'is_category_column': False,
   'is_group_column': False},
  {'name': 'device_id',
   'data_type': 'VARCHAR',
   'required': False,
   'is_category_column': False,
   'is_group_column': False},
  {'name': 'device_category',
   'data_type': 'VARCHAR',
   'required': True,
   'allow_missing': True,
   'is_category_column': True,
   'is_group_column': False,
   'permissible_values': ['imv',
    'nippv',
    'cpap',
    'hfnc',
    'face_mask',
    'trach_collar',
    't_piece',
    'nasal_cannula',
    'room_air',
    'other']},
  {'name': 'vent_brand_name',
   'data_t

## 1. `respiratory_support` — abbreviations + snake_case

`device_category` and `mode_category` carry the hardest renames (`High Flow NC` → `hfnc`,
`Assist Control-Volume Control` → `acvc`, `Pressure Support/CPAP` → `ps_or_cpap`).

In [ ]:
rs = co.respiratory_support.df
rs_30, rs_report = crosswalk_table_2_1_to_3_0(rs, "respiratory_support")
rs_report
rs_30
# for col in ["device_category", "mode_category"]:
#     before = sorted(rs[col].dropna().unique())
#     mapping = {v: normalize_category_value(v) for v in before}  # display only
#     pairs = pd.DataFrame({
#         "2.1 value": before,
#         "3.0 value": [rs_30.loc[rs[col] == v, col].iloc[0] if (rs[col] == v).any() else None for v in before],
#     })
#     print(f"\n=== {col} ===")
#     print(pairs.to_string(index=False))

# print("\nReport is_complete:", rs_report["is_complete"])
# print("device_category values converted:", rs_report["columns"]["device_category"]["n_values_converted"])

## 2. `hospitalization.discharge_category` — curated renames

`Long Term Care Hospital (LTACH)` → `ltach`, `Skilled Nursing Facility (SNF)` → `snf`,
`Psychiatric Hospital` → `mental_health_hosp`, `Against Medical Advice (AMA)` → `ama`.

In [ ]:
hosp = co.hospitalization.df
hosp_30, hosp_report = crosswalk_table_2_1_to_3_0(hosp, "hospitalization")
hosp_30
# hosp_report
# before = sorted(hosp["discharge_category"].dropna().unique())
# out = pd.DataFrame({
#     "2.1 value": before,
#     "3.0 value": [hosp_30.loc[hosp["discharge_category"] == v, "discharge_category"].iloc[0] for v in before],
# })
# print(out.to_string(index=False))
# print("\nis_complete:", hosp_report["is_complete"])

## 3. `patient` — demographics snake_cased

In [ ]:
pat = co.patient.df
pat_30, pat_report = crosswalk_table_2_1_to_3_0(pat, "patient")
for col in ["race_category", "sex_category", "ethnicity_category"]:
    if col in pat.columns:
        b = sorted(pat[col].dropna().unique())
        a = [pat_30.loc[pat[col] == v, col].iloc[0] for v in b]
        print(f"{col}: " + ", ".join(f"{x!r}->{y!r}" for x, y in zip(b, a)))

## 4. The change report

Every call returns a structured report: per-column `n_values_converted`, `ambiguous`
(left-unchanged 1→many), `unresolved` (produced a value not in the 3.0 list), and an
overall `is_complete`.

In [ ]:
print(json.dumps(rs_report, indent=2, default=str))

## 5. Edge cases — ambiguous & removed values (crafted sample)

The demo data doesn't contain these, so we craft a tiny frame to show how the crosswalk
flags values it will **not** silently convert:
- `albumin` → 1→many (`albumin_5` / `albumin_25`): left unchanged, reported as **ambiguous**.
- `csf` lab specimen: **removed** in 3.0, no equivalent: left unchanged, reported.

In [ ]:
# Ambiguous 1->many: albumin in medication_admin_continuous
med = pd.DataFrame({"hospitalization_id": ["H1", "H2"],
                    "med_category": ["albumin", "norepinephrine"]})
med_30, med_report = crosswalk_table_2_1_to_3_0(med, "medication_admin_continuous")
print("med_category after:", list(med_30["med_category"]), "(albumin left unchanged)")
print("ambiguous:", json.dumps(med_report["columns"]["med_category"]["ambiguous"], default=str))

# Removed value: csf lab specimen
labs = pd.DataFrame({"hospitalization_id": ["H1", "H2", "H3"],
                     "lab_specimen_category": ["blood/plasma/serum", "csf", "urine"]})
labs_30, labs_report = crosswalk_table_2_1_to_3_0(labs, "labs")
print("\nlab_specimen_category:", list(labs["lab_specimen_category"]), "->", list(labs_30["lab_specimen_category"]))
print("is_complete:", labs_report["is_complete"])
print("ambiguous:", json.dumps(labs_report["columns"]["lab_specimen_category"]["ambiguous"], default=str))

## 6. Non-mutating, and the normalizer

The input frame is never modified, and `normalize_category_value` is the deterministic
transform used for everything that isn't a curated rename.

In [ ]:
rs_before = co.respiratory_support.df.copy()
_ = crosswalk_table_2_1_to_3_0(co.respiratory_support.df, "respiratory_support")
print("input unchanged after crosswalk:", co.respiratory_support.df.equals(rs_before))

for v in ["IMV", "Non-Hispanic", "l&d", "DNR/DNI", "Mobility/Activity", "pulmonary vasodilators (IV)"]:
    print(f"  {v!r:38} -> {normalize_category_value(v)!r}")

In [23]:
from clifpy import RespiratorySupport

# Validate already-3.0 data against the CLIF 3.0 schema
rs = RespiratorySupport.from_file(
    data_directory="/Users/dema/WD/clifpy/clifpy/data/clif_demo",
    filetype="parquet",
    timezone="US/Central",
    clif_version="3.0",
)
rs_30, rs_report = crosswalk_table_2_1_to_3_0(rs.df, "respiratory_support")
rs.validate()

📢 Initialized respiratory_support table
📢 Data directory: /Users/dema/WD/clifpy/clifpy/data/clif_demo
📢 File type: parquet
📢 Timezone: US/Central
📢 Output directory: /Users/dema/WD/clifpy/examples/output
📢 Loaded schema for 'respiratory_support' (CLIF 3.0)
📢 Loaded outlier configuration
📢 Starting validation
📢 Running schema validation
📢 validate_dataframe: starting validation for table 'respiratory_support'
📢 Running conformance checks for table 'respiratory_support' using polars backend
📢 Conformance checks complete for 'respiratory_support': 6 passed, 0 failed
📢 Running completeness checks for table 'respiratory_support' using polars backend
📢 check_missingness: table 'respiratory_support' — Column 'flow_rate_set' has 89.38% missing values
📢 check_missingness: table 'respiratory_support' — Column 'plateau_pressure_obs' has 84.42% missing values
📢 check_missingness: table 'respiratory_support' — Column 'peak_inspiratory_pressure_obs' has 60.0% missing values
📢 check_missingness: tabl

In [26]:
rs.df = rs_30
rs.validate()

📢 Starting validation
📢 Running schema validation
📢 validate_dataframe: starting validation for table 'respiratory_support'
📢 Running conformance checks for table 'respiratory_support' using polars backend
📢 Conformance checks complete for 'respiratory_support': 6 passed, 0 failed
📢 Running completeness checks for table 'respiratory_support' using polars backend
📢 check_missingness: table 'respiratory_support' — Column 'flow_rate_set' has 89.38% missing values
📢 check_missingness: table 'respiratory_support' — Column 'plateau_pressure_obs' has 84.42% missing values
📢 check_missingness: table 'respiratory_support' — Column 'peak_inspiratory_pressure_obs' has 60.0% missing values
📢 check_missingness: table 'respiratory_support' — Column 'peep_obs' has 85.08% missing values
📢 check_missingness: table 'respiratory_support' — Column 'minute_vent_obs' has 58.77% missing values
📢 check_missingness: table 'respiratory_support' — Column 'mean_airway_pressure_obs' has 59.34% missing values
📢 che